# 3. RAG Pipeline

**Mục tiêu:**
- Tạo **POI embeddings** từ mô tả ngữ nghĩa (tên, category, city, thông tin bổ sung)
- Xây **vector index** để tìm kiếm POI phù hợp nhanh
- Với mỗi user, tổng hợp **user profile vector** từ lịch sử ghé thăm
- Retrieve **top-k POI** phù hợp nhất → tạo **context** cho LLM
- Lưu context đã chuẩn bị vào file JSON

**Cải tiến so với bài báo:** Bài báo gốc chỉ dùng `int_u(c)` dựa trên category. RAG bổ sung semantic similarity — user thích POI theo ngữ nghĩa mô tả thực tế, không chỉ category label.

**Thư viện cần cài:**


In [1]:
!pip install pandas numpy torch

In [2]:
!git clone https://github.com/DuongThai2712/smart-tourism

Cloning into 'smart-tourism'...
remote: Enumerating objects: 1430, done.
remote: Counting objects: 100% (1430/1430), done.
remote: Compressing objects: 100% (1291/1291), done.
remote: Total 1430 (delta 444), reused 1084 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (1430/1430), 18.12 MiB | 6.02 MiB/s, done.
Resolving deltas: 100% (444/444), done.


In [3]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path


project_root = Path("/content/smart-tourism")

raw_path    = project_root / "Datasets" / "Raw"
gf_path     = project_root / "Datasets" / "GraphFrames"   # output Tầng 1
rag_path    = project_root / "Datasets" / "RAG"
rag_path.mkdir(parents=True, exist_ok=True)

# Load từ Parquet (output Tầng 1)
poi_df        = pd.read_parquet(gf_path / "vertices")
pagerank_df   = pd.read_parquet(gf_path / "pagerank")
user_int_df   = pd.read_parquet(gf_path / "user_interest")
dur_df        = pd.read_parquet(gf_path / "poi_duration")
profit_df     = pd.read_parquet(gf_path / "poi_profit")


In [5]:
visit_list = []
for city_dir in sorted(raw_path.iterdir()):
    if not city_dir.is_dir():
        continue
    f = city_dir / "touristsVisits.csv"
    if f.exists():
        df = pd.read_csv(f)
        df["city"] = city_dir.name
        visit_list.append(df)
visit_pdf = pd.concat(visit_list, ignore_index=True)

print(f"POI        : {len(poi_df):,} rows")
print(f"PageRank   : {len(pagerank_df):,} rows")
print(f"UserInterest: {len(user_int_df):,} rows")
print(f"Visits     : {len(visit_pdf):,} rows")

POI        : 444 rows
PageRank   : 444 rows
UserInterest: 15,775 rows
Visits     : 265,554 rows


## POI text descriptions for embedding

In [6]:
# Kết hợp thông tin POI thành câu mô tả ngữ nghĩa
# Mẫu: "Camp Nou is a Sport point of interest in Barcelona at coordinates 41.38, 2.12."

poi_enriched = poi_df.merge(
    pagerank_df[["id", "pagerank"]],
    on="id", how="left"
).merge(
    dur_df[["city", "poiID", "dur_avg_min", "pop_count"]],
    on=["city", "poiID"], how="left"
)

poi_enriched["dur_avg_min"]  = poi_enriched["dur_avg_min"].fillna(30)
poi_enriched["pop_count"]    = poi_enriched["pop_count"].fillna(1)
poi_enriched["pagerank"] = poi_enriched["pagerank"].fillna(0)

In [7]:
def build_poi_description(row) -> str:
    name     = row["poiName"].replace("_", " ")
    category = row["category"]
    city     = row["city"]
    dur      = int(row["dur_avg_min"])
    pop      = int(row["pop_count"])
    lat      = round(row["lat"], 4)
    lon      = round(row["lon"], 4)
    return (
        f"{name} is a {category} attraction in {city}. "
        f"Visitors typically spend {dur} minutes here. "
        f"It has been visited {pop} times (location: {lat}, {lon})."
    )

poi_enriched["description"] = poi_enriched.apply(build_poi_description, axis=1)

print("Ví dụ mô tả POI:")
for desc in poi_enriched["description"].head():
    print(" •", desc)

Ví dụ mô tả POI:
 • Camp Nou is a Sport attraction in Barcelona. Visitors typically spend 27336 minutes here. It has been visited 29 times (location: 41.3809, 2.1228).
 • Park Guell is a Park attraction in Barcelona. Visitors typically spend 2464 minutes here. It has been visited 85 times (location: 41.4145, 2.1527).
 • Placa de Catalunya is a Shopping attraction in Barcelona. Visitors typically spend 39047 minutes here. It has been visited 127 times (location: 41.387, 2.17).
 • Casa Batllo is a Education attraction in Barcelona. Visitors typically spend 12020 minutes here. It has been visited 167 times (location: 41.3916, 2.1648).
 • La Maquinista is a Shopping attraction in Barcelona. Visitors typically spend 30 minutes here. It has been visited 1 times (location: 41.4426, 2.2001).


## Tạo embeddings bằng Transformers (chạy bằng cuda)

In [8]:
from transformers import AutoTokenizer, AutoModel

# Load model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def mean_pool(token_embeddings, attention_mask):
    #Mean pooling — lấy trung bình các token có attention = 1
    mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )
    return (
        torch.sum(token_embeddings * mask_expanded, dim=1)
        / torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    )


def encode_texts(texts: list[str], batch_size: int = 64) -> torch.Tensor:
    """
    Encode danh sách text thành normalized embedding vectors.
    Trả về Tensor shape (N, hidden_dim) trên CPU.
    """
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]

            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            ).to(DEVICE)

            outputs = model(**encoded)

            # Mean pool trên last hidden state
            embeddings = mean_pool(
                outputs.last_hidden_state,
                encoded["attention_mask"],
            )

            # L2 normalize → cosine similarity = dot product
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

            all_embeddings.append(embeddings.cpu())

            if (i // batch_size + 1) % 5 == 0:
                print(f"  Encoded {min(i+batch_size, len(texts))}/{len(texts)}")

    return torch.cat(all_embeddings, dim=0)

In [10]:
descriptions = poi_enriched["description"].tolist()
poi_embeddings_tensor = encode_texts(descriptions, batch_size=64)
poi_embeddings = poi_embeddings_tensor.numpy()   # chuyển sang numpy để FAISS dùng

print(f"Embeddings shape: {poi_embeddings.shape}")

  Encoded 320/444
Embeddings shape: (444, 384)


In [11]:
# Lưu embeddings
np.save(rag_path / "poi_embeddings.npy", poi_embeddings)
poi_enriched.to_parquet(rag_path / "poi_enriched.parquet", index=False)
print("Đã lưu poi_embeddings.npy và poi_enriched.parquet")

Đã lưu poi_embeddings.npy và poi_enriched.parquet


## Torch Vector Index

Vector index bằng PyTorch thuần.Dùng batched dot product (= cosine similarity sau khi chuẩn hoá).

In [12]:
class TorchVectorIndex:
    def __init__(self, embeddings: torch.Tensor):
        # embeddings: (N, D), đã normalize
        self.index   = embeddings.to(DEVICE)   # đưa lên GPU nếu có
        self.n, self.d = embeddings.shape

    def search(self, query: torch.Tensor, k: int):
        """
        query : (1, D) hoặc (Q, D), đã normalize
        Trả về (scores, indices) — giống API của FAISS
        """
        query = query.to(DEVICE)
        # Dot product = cosine similarity (vì đã normalize cả hai)
        scores = torch.matmul(query, self.index.T)   # (Q, N)
        top_scores, top_idx = torch.topk(scores, k=min(k, self.n), dim=1)
        return top_scores.cpu().numpy(), top_idx.cpu().numpy()

    def save(self, path: str):
        torch.save(self.index.cpu(), path)

    @classmethod
    def load(cls, path: str):
        embeddings = torch.load(path, map_location="cpu")
        return cls(embeddings)



In [13]:
torch_index = TorchVectorIndex(poi_embeddings_tensor)
torch_index.save(str(rag_path / "poi_index.pt"))
print(f"TorchVectorIndex built: {torch_index.n} vectors, dim={torch_index.d}")

TorchVectorIndex built: 444 vectors, dim=384


## Tạo User Profile Vector từ lịch sử ghé thăm

Tổng hợp user profile:

    - visited_pois: danh sách POI đã ghé
    - top_categories: category ưa thích theo int_u(c)
    - profile_vector: embedding vector trung bình của các POI đã ghé
    - budget_min: ngân sách thời gian

In [14]:
def build_user_profile(
    user_id: str, city: str,
    visit_pdf, poi_enriched,
    user_int_df, embedder,
    budget_min: float = 480,
) -> dict:
    # POI đã ghé của user trong city
    user_visits = visit_pdf[
        (visit_pdf["userID"] == user_id) & (visit_pdf["city"] == city)
    ]["poiID"].unique().tolist()

    # Category ưa thích
    user_cats = (
        user_int_df[
            (user_int_df["userID"] == user_id) & (user_int_df["city"] == city)
        ]
        .sort_values("int_u_c", ascending=False)
        .head(3)
    )
    top_cats = user_cats[["poiTheme", "int_u_c"]].to_dict("records")

    # Profile vector: trung bình embedding các POI đã ghé
    visited_descs = poi_enriched[
        (poi_enriched["city"] == city) &
        (poi_enriched["poiID"].isin(user_visits))
    ]["description"].tolist()

    if visited_descs:
        vecs = encode_texts(visited_descs).numpy()
        profile_vec = vecs.mean(axis=0)
        # Normalize lại sau mean
        norm = np.linalg.norm(profile_vec)
        if norm > 0:
            profile_vec = profile_vec / norm
    else:
        # Không có lịch sử: dùng embedding của category ưa thích
        cat_desc = f"A tourist interested in {', '.join([c['poiTheme'] for c in top_cats])} attractions in {city}."
        profile_vec = encode_texts([cat_desc]).numpy()[0]

    # Tên các POI đã ghé
    visited_names = poi_enriched[
        (poi_enriched["city"] == city) &
        (poi_enriched["poiID"].isin(user_visits))
    ]["poiName"].apply(lambda x: x.replace("_", " ")).tolist()

    return {
        "user_id":       user_id,
        "city":          city,
        "visited_pois":  visited_names,
        "top_categories": top_cats,
        "profile_vector": profile_vec,
        "budget_min":    budget_min,
    }

print("Hàm build_user_profile đã sẵn sàng.")

Hàm build_user_profile đã sẵn sàng.


## Hàm Retrieve top-k POI
Tìm top-k POI phù hợp nhất với user profile trong cùng city. Trả về DataFrame với cột:  ```poiName, category, similarity, dur_avg_min, description```

In [15]:
from urllib.parse import unquote
import re

def _norm(name: str) -> str:
    """Normalize nội bộ để so sánh tên POI."""
    name = unquote(str(name))
    name = name.replace("_", " ").strip().lower()
    name = re.sub(r"[().',\-]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

def retrieve_top_k_pois(profile: dict, torch_index: TorchVectorIndex,
    poi_enriched: pd.DataFrame, dur_df: pd.DataFrame, k: int = 10,
    exclude_visited: bool = True,
) -> pd.DataFrame:
    city      = profile["city"]
    query_vec = torch.tensor(
        profile["profile_vector"], dtype=torch.float32
    ).unsqueeze(0)   # (1, D)

    # Tìm k*3 để có đủ sau khi filter city + visited
    scores, indices = torch_index.search(query_vec, k * 3)

    candidates = poi_enriched.iloc[indices[0]].copy()
    candidates["similarity"] = scores[0]

    # Filter cùng city
    candidates = candidates[candidates["city"] == city]

    # Loại POI đã ghé — dùng normalize thống nhất cả 2 phía
    if exclude_visited and profile["visited_pois"]:
        visited_norm = set(_norm(v) for v in profile["visited_pois"])
        candidates   = candidates[
            ~candidates["poiName"].apply(_norm).isin(visited_norm)
        ]

    # Merge duration
    candidates = candidates.merge(
        dur_df[["city", "poiID", "dur_avg_min"]],
        on=["city", "poiID"], how="left"
    )
    candidates["dur_avg_min"] = (
        candidates["dur_avg_min_y"]
        .fillna(candidates["dur_avg_min_x"])
        .fillna(30)
    )

    return candidates.head(k)[[
        "id", "poiName", "category", "similarity",
        "dur_avg_min", "lat", "lon", "description"
    ]].reset_index(drop=True)

print("Hàm retrieve_top_k_pois đã sẵn sàng.")

Hàm retrieve_top_k_pois đã sẵn sàng.


## Chạy thử RAG

In [18]:
DEMO_CITY = "Athens"

# Load lại Torch index
torch_index = TorchVectorIndex.load(str(rag_path / "poi_index.pt"))
poi_enriched = pd.read_parquet(rag_path / "poi_enriched.parquet")

# Lấy user mẫu
demo_users = visit_pdf[
    visit_pdf["city"] == DEMO_CITY
]["userID"].value_counts().head(3).index.tolist()

for demo_user in demo_users:
    profile = build_user_profile(user_id=demo_user, city=DEMO_CITY,
        visit_pdf=visit_pdf, poi_enriched=poi_enriched,
        user_int_df=user_int_df, embedder=None,   # không dùng nữa, encode_texts() dùng trực tiếp
        budget_min= 480,
    )

    retrieved = retrieve_top_k_pois(profile=profile, torch_index=torch_index,
        poi_enriched=poi_enriched, dur_df=dur_df, k= 8,
    )

    print(f"User: {demo_user} | City: {DEMO_CITY}")
    print(f"Đã ghé: {profile['visited_pois']}")
    print(f"Top categories: {profile['top_categories']}")
    print(f"\nTop-8 POI đề xuất:")
    print(retrieved[["poiName", "category", "similarity", "dur_avg_min"]].to_string(index=False))

User: 87974483@N02 | City: Athens
Đã ghé: ['National Archaeological Museum', 'Syntagma (Constitution) Square']
Top categories: [{'poiTheme': 'Museum', 'int_u_c': 0.002575965071114844}, {'poiTheme': 'Historical', 'int_u_c': 0.00023632741802381455}]

Top-8 POI đề xuất:
                            poiName   category  similarity  dur_avg_min
         National_Historical_Museum     Museum    0.833492      4412.88
                   Acropolis_Museum     Museum    0.804858     24358.05
                Acropolis_of_Athens Historical    0.801235      6421.91
        Municipal_Gallery_of_Athens     Museum    0.796177     60277.20
        Numismatic_Museum_of_Athens     Museum    0.794723        46.01
   Athens_University_History_Museum     Museum    0.787221     20315.43
                  Athens_War_Museum     Museum    0.764729        14.54
Museum_of_Illusions_Athens_-_Greece     Museum    0.745410     44527.92
User: 46981478@N00 | City: Athens
Đã ghé: ['Syntagma (Constitution) Square']
Top cat

## Tạo context JSON cho LLM

In [19]:
import json

def build_llm_context(
    profile: dict,
    retrieved_pois: pd.DataFrame,
    graphframes_itinerary: list = None,  # kết quả BFS từ Tầng 1
) -> dict:
    """
    Tạo context hoàn chỉnh cho LLM:
    - user profile (lịch sử, sở thích, ngân sách)
    - top-k POI retrieved (ngữ nghĩa)
    - itinerary gợi ý từ GraphFrames (để LLM có tham chiếu)
    """
    context = {
        "user_id":   profile["user_id"],
        "city":      profile["city"],
        "budget_minutes": profile["budget_min"],
        "user_profile": {
            "visited_pois":   profile["visited_pois"],
            "top_categories": profile["top_categories"],
        },
        "retrieved_pois": [
            {
                "name":        row["poiName"].replace("_", " "),
                "category":    row["category"],
                "similarity":  round(float(row["similarity"]), 4),
                "avg_visit_min": round(float(row["dur_avg_min"]), 1),
                "description": row["description"],
            }
            for _, row in retrieved_pois.iterrows()
        ],
    }

    if graphframes_itinerary:
        context["graphframes_suggestion"] = [
            {
                "rank":       rank + 1,
                "path":       [p.replace("_", " ") for p in path],
                "profit":     round(profit, 4),
                "total_time_min": round(time_min, 1),
            }
            for rank, (path, profit, time_min) in enumerate(graphframes_itinerary)
        ]

    return context


print("Hàm build_llm_context đã sẵn sàng.")

Hàm build_llm_context đã sẵn sàng.


In [20]:
import time
BUDGET_MIN      = 480  # 8 giờ

# Load BFS itineraries từ main.ipynb
bfs_itineraries_path = project_root / "Datasets" / "GraphFrames" / "bfs_itineraries_all.csv"
if bfs_itineraries_path.exists():
    bfs_df = pd.read_csv(bfs_itineraries_path)
    print(f"Loaded BFS itineraries: {len(bfs_df):,} rows")
else:
    bfs_df = None
    print("Không tìm thấy BFS itineraries, bỏ qua graphframes_suggestion.")

# Reload để chắc chắn dùng bản mới nhất
torch_index  = TorchVectorIndex.load(str(rag_path / "poi_index.pt"))
poi_enriched = pd.read_parquet(rag_path / "poi_enriched.parquet")

# Lấy tất cả thành phố và tất cả user có dữ liệu
all_cities = sorted(poi_enriched["city"].unique().tolist())
print(f"Tổng số thành phố: {len(all_cities)} → {all_cities}")

all_contexts = []
total_start  = time.time()



Loaded BFS itineraries: 5,485 rows
Tổng số thành phố: 13 → ['Athens', 'Barcelona', 'Budapest', 'Edinburgh', 'Glasgow', 'London', 'Madrid', 'Melbourne', 'NewDelhi', 'Osaka', 'Perth', 'Toronto', 'Vienna']


In [21]:
# ── TÁCH TRAIN VISITS (dùng 80% đầu để build profile) ───────────────────────
from urllib.parse import unquote

def get_train_visits(visit_pdf, train_ratio=0.8):
    """Trả về DataFrame chỉ gồm phần train (80% visit đầu) cho mỗi (user, city)."""
    visit_sorted = visit_pdf.sort_values("dateTaken")
    def keep_train(group):
        n = len(group)
        split = max(1, int(n * train_ratio))
        return group.iloc[:split]
    return (
        visit_sorted
        .groupby(["userID", "city"], group_keys=False)
        .apply(keep_train)
    )

train_visit_pdf = get_train_visits(visit_pdf)
print(f"Train visits: {len(train_visit_pdf):,} / {len(visit_pdf):,} tổng")

Train visits: 208,882 / 265,554 tổng


/tmp/ipykernel_2003/1113633044.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_train)


In [22]:
for city in all_cities:
    city_start = time.time()

    # Tất cả user có ghé ít nhất 1 POI trong city này
    city_users = train_visit_pdf[train_visit_pdf["city"] == city]["userID"].unique().tolist()
    print(f"\n── {city}: {len(city_users)} users ──")

    for u_idx, user_id in enumerate(city_users):
        profile = build_user_profile(user_id=user_id, city=city,
            visit_pdf=train_visit_pdf, poi_enriched=poi_enriched,
            user_int_df=user_int_df,embedder=None,
            budget_min=BUDGET_MIN)

        retrieved = retrieve_top_k_pois(profile=profile, torch_index=torch_index,
                                        poi_enriched=poi_enriched, dur_df=dur_df, k= 20)

        # Lấy BFS itineraries của user này nếu có
        gf_itinerary = None
        if bfs_df is not None:
            user_bfs = bfs_df[
                (bfs_df["city"] == city) & (bfs_df["userID"] == user_id)
            ].sort_values("rank")
            if not user_bfs.empty:
                gf_itinerary = [
                    (row["path_names"].split(" → "), row["total_profit"], row["total_time_min"])
                    for _, row in user_bfs.iterrows()
                ]

        context = build_llm_context(
            profile               = profile,
            retrieved_pois        = retrieved,
            graphframes_itinerary = gf_itinerary,
        )
        all_contexts.append(context)

        # Progress mỗi 100 user
        if (u_idx + 1) % 100 == 0 or (u_idx + 1) == len(city_users):
            print(f"  {u_idx+1:>4}/{len(city_users)} users  "
                  f"({time.time()-city_start:.0f}s elapsed)")

    print(f"  ✓ {city} hoàn thành: {len(city_users)} users, "
          f"{time.time()-city_start:.1f}s")

# Lưu tất cả context
context_file = rag_path / "llm_contexts.json"
with open(context_file, "w", encoding="utf-8") as f:
    json.dump(all_contexts, f, ensure_ascii=False, indent=2)

print(f"\nĐã lưu {len(all_contexts):,} contexts -> {context_file}")
print(f"Wall time tổng: {time.time()-total_start:.1f}s")


── Athens: 296 users ──
   100/296 users  (5s elapsed)
   200/296 users  (10s elapsed)
   296/296 users  (15s elapsed)
  ✓ Athens hoàn thành: 296 users, 15.5s

── Barcelona: 789 users ──
   100/789 users  (5s elapsed)
   200/789 users  (11s elapsed)
   300/789 users  (26s elapsed)
   400/789 users  (30s elapsed)
   500/789 users  (37s elapsed)
   600/789 users  (41s elapsed)
   700/789 users  (47s elapsed)
   789/789 users  (52s elapsed)
  ✓ Barcelona hoàn thành: 789 users, 51.6s

── Budapest: 935 users ──
   100/935 users  (5s elapsed)
   200/935 users  (11s elapsed)
   300/935 users  (16s elapsed)
   400/935 users  (22s elapsed)
   500/935 users  (27s elapsed)
   600/935 users  (33s elapsed)
   700/935 users  (38s elapsed)
   800/935 users  (43s elapsed)
   900/935 users  (50s elapsed)
   935/935 users  (51s elapsed)
  ✓ Budapest hoàn thành: 935 users, 51.3s

── Edinburgh: 1454 users ──
   100/1454 users  (5s elapsed)
   200/1454 users  (11s elapsed)
   300/1454 users  (19s elapsed)

In [23]:
# Preview context đầu tiên
contx = all_contexts[:100]
for ctx in contx:
    print(f"User    : {ctx['user_id']}")
    print(f"City    : {ctx['city']}")
    print(f"Đã ghé  : {ctx['user_profile']['visited_pois']}")
    print(f"Top cats: {ctx['user_profile']['top_categories']}")
    print(f"\nPOI retrieved (maximum 10):")
    for i, poi in enumerate(ctx["retrieved_pois"][:10], 1):
        print(f"  {i:2d}. [{poi['category']:12s}] {poi['name']:35s} "
              f"sim={poi['similarity']:.3f}  ~{poi['avg_visit_min']:.0f}min")

User    : 100721644@N08
City    : Athens
Đã ghé  : ['Museum of Illusions Athens - Greece']
Top cats: [{'poiTheme': 'Museum', 'int_u_c': 6.737346222165587e-06}]

POI retrieved (maximum 10):
   1. [Museum      ] National Archaeological Museum      sim=0.813  ~63808min
   2. [Museum      ] Municipal Gallery of Athens         sim=0.806  ~60277min
   3. [Museum      ] Acropolis Museum                    sim=0.802  ~24358min
   4. [Museum      ] Athens University History Museum    sim=0.783  ~20315min
   5. [Museum      ] The Jewish Museum of Greece         sim=0.779  ~2405min
   6. [Museum      ] National Historical Museum          sim=0.777  ~4413min
   7. [Museum      ] Numismatic Museum of Athens         sim=0.766  ~46min
   8. [Museum      ] Athens War Museum                   sim=0.765  ~14min
   9. [Museum      ] Museums of the City of Athens Vouros Eutaxias Foundation sim=0.759  ~70317min
  10. [Museum      ] Hellenic Motor Museum               sim=0.758  ~1088min
User    : 100768432

## Push lên Git

In [24]:
%cd /content/smart-tourism

/content/smart-tourism


In [25]:
!git remote -v

origin	https://github.com/DuongThai2712/smart-tourism (fetch)
origin	https://github.com/DuongThai2712/smart-tourism (push)


In [26]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Datasets/RAG/llm_contexts.json

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
!git add Datasets/RAG

In [ ]:
!git config --global user.email  #"you@example.com"
!git config --global user.name #"Your Name"
!git commit -m "Add new RAG dataset"

[main 62af2b4] Add new RAG dataset
 1 file changed, 275792 insertions(+), 278168 deletions(-)


In [33]:
!git pull origin main

From https://github.com/DuongThai2712/smart-tourism
 * branch            main       -> FETCH_HEAD
Already up to date.


In [34]:
!git push https://DuongThai2712:@github.com/DuongThai2712/smart-tourism.git main

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 460.84 KiB | 300.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
remote: warning: See https://gh.io/lfs for more information.
remote: warning: File Datasets/RAG/llm_contexts.json is 78.52 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.
To https://github.com/DuongThai2712/smart-tourism.git
   febb254..62af2b4  main -> main
